# MonIA — Test connexion Kaggle ↔ GitHub
Ce notebook vérifie le GPU, Internet et le secret `GITHUB_TOKEN` sans jamais afficher le token.

In [ ]:
import requests, torch
from kaggle_secrets import UserSecretsClient

print('=== MonIA Kaggle readiness ===')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NON DISPONIBLE')
print('GPU count:', torch.cuda.device_count())

try:
    r=requests.get('https://api.github.com/repos/vartcom38-collab/marion-lucas-game',timeout=20)
    print('Internet/GitHub public:', 'OK' if r.ok else f'ERREUR HTTP {r.status_code}')
except Exception as e:
    print('Internet/GitHub public: ERREUR', type(e).__name__)

try:
    token=UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    token=None

if not token:
    print('GITHUB_TOKEN: INTROUVABLE')
else:
    headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json'}
    r=requests.get('https://api.github.com/repos/vartcom38-collab/marion-lucas-game',headers=headers,timeout=20)
    if not r.ok:
        print('GITHUB_TOKEN: REFUSÉ', r.status_code)
    else:
        perms=(r.json().get('permissions') or {})
        print('GITHUB_TOKEN: OK')
        print('Repo push permission:', bool(perms.get('push')))
        print('Repo pull permission:', bool(perms.get('pull')))
        print('READY:', bool(torch.cuda.is_available() and perms.get('push')))
